[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/proyecto/proyecto.ipynb)

# Proyecto final: analistas de MiBici
**DAMMG · Módulo VI** · equipos de 2 o 3 · entrega: **domingo 22 de noviembre, 23:59** (tolerancia hasta el miércoles 25) · 30 % del módulo

**Equipo:** _(nombres)_

**Qué entregan.** Este notebook, que debe correr de principio a fin con *Reiniciar y ejecutar todo*, y un reporte de **2 páginas** en PDF (carta, 11 pt o más, figuras incluidas en las 2 páginas, con una línea final de quién hizo qué). Una sola entrega por equipo. Enunciado completo: [README del proyecto](https://github.com/HesusG/dammg-supervisado/blob/main/notebooks/proyecto/README.md).

**Equipo en Colab.** Una persona hace *Archivo*, opción *Guardar una copia en Drive* y la comparte con el resto como **editor**; trabajen todos sobre esa copia (no al mismo tiempo en la misma celda). Registren el equipo en Moodle a más tardar el **martes 10 de noviembre**.

**Fecha.** Entreguen a tiempo **aunque el notebook todavía no corra completo**: en ese caso se descuenta 1 punto y tienen 48 h para corregirlo. Después del miércoles 25, 23:59, no se recibe.

**Reglas que más se olvidan.** Test se usa una vez, al final. Todo lo que se aprende de los datos se ajusta solo con train. En MiBici están prohibidas como entrada `departures`, `arrivals` y `net_flow` (son de la hora *t*). Pueden usar IA para estudiar o depurar; el análisis y la redacción son del equipo.

**Cómo usar esta plantilla.** Cada sección corresponde a un criterio de la rúbrica (C1 a C6). Todas las celdas ya corren con el caso MiBici del módulo, así que pueden empezar por ahí. Lo que les toca escribir está marcado con _(escribe aquí)_ y con comentarios `# EQUIPO:`.

| Criterio | Peso | Sección |
|---|---|---|
| C1. Problema y datos | 10 % | 1 |
| C2. Partición honesta y baseline | 20 % | 2 |
| C3. Comparación de modelos (≥ 3, hiperparámetros en validación) | 20 % | 3 |
| C4. Métrica y umbral justificados | 20 % | 4 |
| C5. Lectura con la teoría (sesgo/varianza y sesgo inductivo) | 20 % | 5 |
| C6. Limitaciones y riesgos | 10 % | 6 |

## Las preguntas que responden
Son el equipo de datos de MiBici. Responden **preguntas de negocio** con los modelos del módulo, siempre con los datos de MiBici.

**Preguntas de negocio**
1. **Mover bicis de noche.** ¿Cuántas bicis saldrán mañana a las 8 a.m. en cada estación, y con qué error? ¿Le ganan a "lo mismo que la semana pasada"? (regresión, MAE y RMSE)
2. **Alertas con presupuesto.** Con $N$ viajes de camión al día, ¿a qué estaciones y horas mandan alerta? ¿Cuántos desbalances atrapan y cuántos viajes son en balde? (clasificación, umbral, precision y recall)
3. **¿Dónde falla el modelo?** ¿En qué estaciones, horas o días se equivoca más? ¿Qué le recomiendan a operaciones? (errores por grupo, características derivadas)
4. **¿Qué modelo ponemos en producción?** Comparen al menos tres modelos del módulo (lineal o logística, árbol o bosque, SVM) y recomienden uno, con su métrica.
5. **¿Sigue sirviendo después de las vacaciones?** El modelo entrenado hasta junio, ¿funciona igual en agosto y septiembre?

**Teoría**
- **T1.** Si predicen siempre la misma constante $c$, demuestren que $\frac{1}{m}\sum_i (c - y^{(i)})^2$ es mínimo cuando $c = \bar y$.
  _Pista: repasen la derivada del Módulo III y el costo J de la sesión 1 (bloque B)._
- **T2.** Con un presupuesto fijo de $k$ alertas, muestren que $\text{precision} = TP/k$ y que $\text{recall} = \text{precision}\cdot k / P$, con $P$ los desbalances reales. ¿Qué quiere decir para el negocio?
  _Pista: repasen la matriz de confusión de la sesión 2; las k alertas son TP + FP._

Las dos tienen andamiaje al final del notebook, en la sección **Teoría (T1 y T2)**.

Las preguntas caen en los criterios de la rúbrica: la 1 y la 2 en C2 a C4; la 3 y la 5 en C5 y C6; la 4 en C3; la teoría, en C5.

## 0 · Preparación

> **Antes de escribir nada**
> 1. *Archivo*, opción *Guardar una copia en Drive* y trabaja en esa copia; si no, al cerrar la pestaña pierdes todo.
> 2. Ejecuta las celdas **en orden**. Si reinicias el entorno o algo falla raro: *Entorno de ejecución*, opción *Reiniciar y ejecutar todo*.
> 3. Escribe dentro de las celdas que ya existen (no copies/pegues celdas): así no pierden la etiqueta que usa el evaluador.
>
> *Palabras en inglés que usamos tal cual:* **baseline** (línea base: el modelo tonto que hay que vencer) · **pipeline** (cadena de preprocesamiento + modelo) · **proxy** (variable sustituta de lo que de verdad queremos medir) · **bootstrap** (remuestreo con reemplazo) · **chat waterfall** (todos escriben su respuesta en el chat y la envían al mismo tiempo) · **precision** y **recall** (los nombres de sklearn para precisión y exhaustividad).

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
import time
from sklearn.metrics import average_precision_score, precision_score, recall_score, confusion_matrix
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

## 1 · Problema y datos (C1)
- **Pregunta de negocio:** _(¿qué decisión se toma con la predicción, quién la toma y cada cuánto?)_
- **Variable objetivo y unidad de predicción:** _(p. ej. "¿habrá flujo neto extremo en esta estación en las próximas 3 h?", una fila = estación-hora)_
- **De dónde salen los datos y qué no dicen:** _(fuente, periodo, licencia; qué no se observa; sesgos de registro)_

In [ ]:
# EQUIPO: si usan sus propios datos, cárguenlos aquí (CSV o parquet) y documenten columnas y periodo.
# Por defecto: el panel MiBici del módulo (40 estaciones, oct 2025–sep 2026, una fila por estación-hora).
df = pd.read_parquet(RUTA_DATOS)
print(df.shape)
print("periodo:", df["hour"].min(), "a", df["hour"].max())
df.head()

**Opcional · panel completo.** El mismo panel MiBici que usa el curso TC5061: 469 estaciones, oct 2025–sep 2026,
4.1 millones de filas (unos 40 MB; en Colab carga en menos de un minuto). Trae columnas extra para comparar con ese curso:
`departures_lag_48h`, `departures_lag_336h`, `municipality` y la etiqueta `risk` (|`future_net`| ≥ 0.5 × capacidad,
prevalencia ≈ 2 %), además de una columna `particion` con la división temporal de TC5061
(train ≤ jun 2026, val = jul, test = ago–sep). Las primeras horas de cada estación tienen lags vacíos (`NaN`).
Con 4 millones de filas, SVC y las rejillas grandes no terminan: submuestreen train o usen modelos lineales y de árboles.

In [ ]:
USAR_PANEL_COMPLETO = False   # EQUIPO: True para trabajar con el panel completo

if USAR_PANEL_COMPLETO:
    RUTA_PANEL = _obtener("mibici_panel_completo.parquet", "37c815f15cd5b8b9713c36705fa8b58746060a5211bd3683cc1befada94cea79")   # misma descarga verificada que la celda L0
    df = pd.read_parquet(RUTA_PANEL)
    print(df.shape, "·", df["station_id"].nunique(), "estaciones")
    print(df["particion"].value_counts())

In [ ]:
# Una mirada a la variable objetivo antes de modelar (EQUIPO: adapten a su problema)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(df["future_net"], bins=60, color=COLORES["w"])
ax.set(xlabel="future_net (salidas − llegadas en [t, t+3h))", ylabel="estaciones-hora", yscale="log",
       title="Distribución de la variable de la que sale la etiqueta")
plt.show()

## 2 · Partición honesta y baseline (C2)
Reglas que la rúbrica revisa:
1. Si hay tiempo, la partición es **temporal** (entrenar con el pasado, validar con lo que sigue, probar con lo último).
2. Todo lo que se "aprende" (umbral del evento, escaladores, imputaciones, codificaciones) se ajusta **solo con train**.
3. Ninguna variable usa información que no tendrías en el momento de predecir (en MiBici: nada de la hora $t$ ni del futuro).
4. Test se usa **una vez**, al final (sección 4b).

In [ ]:
# EQUIPO: definan su partición. En MiBici ya viene: train = oct 2025–jun 2026, val = jul, test = ago–sep.
tr = df[df["particion"] == "train"].copy()
va = df[df["particion"] == "val"].copy()
te = df[df["particion"] == "test"].copy()

# Umbral del evento: se calcula SOLO con train y se congela
UMBRAL = float(tr["future_net"].abs().quantile(0.85))
y_tr, y_va, y_te = ((d["future_net"].abs() >= UMBRAL).astype(int).to_numpy() for d in (tr, va, te))
print(f"umbral del evento = {UMBRAL} · prevalencia train/val/test = {y_tr.mean():.3f} / {y_va.mean():.3f} / {y_te.mean():.3f}")

HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]


def variables(d):
    # EQUIPO: sus variables. Para cada una, pregúntense "¿a qué hora exacta la conocería en producción?".
    return pd.DataFrame({
        "net_lag_24h": d["net_lag_24h"], "net_lag_168h": d["net_lag_168h"], "net_mean_24h": d["net_mean_24h"],
        "abs_net_lag_24h": d["net_lag_24h"].abs(), "abs_net_lag_168h": d["net_lag_168h"].abs(),
        "abs_net_mean_24h": d["net_mean_24h"].abs(),
        "departures_mean_24h": d["departures_mean_24h"], "departures_mean_168h": d["departures_mean_168h"],
        "departures_lag_168h": d["departures_lag_168h"], "capacity": d["capacity"],
        "is_weekend": d["is_weekend"], "pico": d["hour_of_day"].isin(HORAS_PICO).astype(int),
        "hour_of_day": d["hour_of_day"], "day_of_week": d["day_of_week"],
    }, index=d.index)


X_tr, X_va, X_te = variables(tr), variables(va), variables(te)
print("variables:", list(X_tr.columns))

In [ ]:
# Auditoría rápida de fugas: AP en val de cada variable sola. Si una sola le gana a todo, desconfíen.
auditoria = pd.Series({col: max(average_precision_score(y_va, X_va[col]), average_precision_score(y_va, -X_va[col]))
                       for col in X_va.columns}).sort_values(ascending=False)
auditoria.round(3).to_frame("AP val, variable sola")

In [ ]:
# Baselines: el que no hace nada y una regla razonable sin aprendizaje (EQUIPO: elijan la suya y justifíquenla)
puntajes_va = {
    "baseline mayoritario (siempre 0)": np.zeros(len(y_va)),
    "baseline: la semana se repite (|net_lag_168h|)": X_va["abs_net_lag_168h"].to_numpy(),
}
for nombre, s in puntajes_va.items():
    print(f"{nombre:48s} AP val = {average_precision_score(y_va, s):.3f}")

**Justificación de la partición y del baseline:** _(escribe aquí: por qué esta partición evita fugas y por qué este baseline es el que hay que vencer)_

## 3 · Comparación de modelos (C3)
Al menos **tres** modelos del módulo (logística, árbol, bosque, boosting, SVM). Cada uno con sus hiperparámetros elegidos **en validación**: dejen la rejilla en el notebook, no solo el ganador. La plantilla trae una logística con su rejilla de $C$ como ejemplo; agreguen los suyos con el mismo patrón.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

escalador = StandardScaler().fit(X_tr)                 # solo train
Z_tr, Z_va, Z_te = (escalador.transform(X) for X in (X_tr, X_va, X_te))

def prueba(familia, hiper, modelo, escalado):
    # Ajusta en train, mide AP en val y registra la fila de la rejilla
    A_tr, A_va = (Z_tr, Z_va) if escalado else (X_tr, X_va)
    t0 = time.perf_counter()
    modelo.fit(A_tr, y_tr)
    seg = time.perf_counter() - t0
    s_va = modelo.predict_proba(A_va)[:, 1] if hasattr(modelo, "predict_proba") else modelo.decision_function(A_va)
    s_tr = modelo.predict_proba(A_tr)[:, 1] if hasattr(modelo, "predict_proba") else modelo.decision_function(A_tr)
    rejillas.append({"familia": familia, "hiperparámetros": hiper, "AP train": average_precision_score(y_tr, s_tr),
                     "AP val": average_precision_score(y_va, s_va), "segundos": seg})
    return modelo, s_va, escalado


rejillas = []           # una fila por (modelo, hiperparámetros) probados. Si re-ejecutas ESTA celda se reinicia
elegidos = {}           # modelo ganador de cada familia, ya ajustado

# Ejemplo: logística con una rejilla de C (C = 1/λ, ver la celda ⚠ del Lab 2)
mejor = None
for C in [0.01, 0.1, 1, 10]:
    res = prueba("logística", f"C={C}", LogisticRegression(C=C, max_iter=2000), escalado=True)
    if mejor is None or rejillas[-1]["AP val"] > mejor[0]:
        mejor = (rejillas[-1]["AP val"], res)
elegidos["logística"] = mejor[1]

# EQUIPO: agreguen al menos dos familias más con el mismo patrón, cada una en UNA CELDA NUEVA debajo de esta
# (así no borran rejillas al re-ejecutar). Rejillas chicas: 2–3 valores. En Colab gratuito, un bosque o un
# boosting sobre ~254 mil filas tarda de 40 s a 4 min por punto; el SVM con kernel, en una submuestra (Lab 4).
# Por ejemplo:
# from sklearn.ensemble import HistGradientBoostingClassifier
# mejor = None
# for lr in [0.05, 0.1]:
#     res = prueba("boosting", f"lr={lr}", HistGradientBoostingClassifier(learning_rate=lr, max_iter=300,
#                  early_stopping=False, random_state=0), escalado=False)
#     if mejor is None or rejillas[-1]["AP val"] > mejor[0]:
#         mejor = (rejillas[-1]["AP val"], res)
# elegidos["boosting"] = mejor[1]

pd.DataFrame(rejillas).round(3)

In [ ]:
# Tabla comparativa: baselines + el mejor de cada familia, ordenada por AP en validación
for familia, (modelo, s_va, escalado) in elegidos.items():
    puntajes_va[familia] = s_va
comparativa = pd.DataFrame({"AP val": {k: average_precision_score(y_va, s) for k, s in puntajes_va.items()}})
comparativa = comparativa.sort_values("AP val", ascending=False)
if len(elegidos) < 3:
    print(f"⚠ Llevan {len(elegidos)} familia(s) de modelos: la rúbrica pide al menos 3.")
comparativa.round(3)

> **¿Su problema es de regresión?** La plantilla está escrita para clasificación (AP, umbral). Para regresión:
> 1. Cambien `prueba` para que mida el error en validación con `modelo.predict` y `mean_absolute_error` (o RMSE) en lugar de `predict_proba` y AP, y ordenen la comparativa de **menor** a mayor error.
> 2. En la sección 4 (C4) no hay umbral: justifiquen la métrica por el costo del error (MAE si todos los errores cuestan igual por unidad; RMSE si un error grande es desproporcionadamente caro) y reporten test una vez.
> 3. El baseline natural es "la semana se repite" o la media de train, como en el Lab 1.

## 4 · Métrica y umbral (C4)
- **Métrica para elegir modelo:** _(p. ej. AP, si la clase positiva es rara y lo que importa es la calidad de las alertas)_
- **Cómo se fija el umbral del clasificador:** _(presupuesto de alertas, o costo de un falso positivo contra el de un falso negativo; con números)_

Dos formas de fijar el umbral **en validación**. Usen la que corresponda a su problema y justifíquenla.

In [ ]:
def umbral_presupuesto(p, presupuesto):
    # El umbral más bajo t con mean(p >= t) <= presupuesto
    cand = np.unique(p)
    tasa = 1 - np.searchsorted(np.sort(p), cand, side="left") / len(p)
    return float(cand[np.argmax(tasa <= presupuesto)])


def umbral_costo(y, p, costo_fp, costo_fn):
    # El umbral que minimiza costo_fp·FP + costo_fn·FN en los datos dados
    cand = np.unique(p)
    costos = [costo_fp * np.sum((p >= t) & (y == 0)) + costo_fn * np.sum((p < t) & (y == 1)) for t in cand[:: max(1, len(cand) // 500)]]
    return float(cand[:: max(1, len(cand) // 500)][int(np.argmin(costos))])


candidato = [k for k in comparativa.index if k in elegidos][0]        # el mejor modelo (no baseline) en val
s_va_cand = puntajes_va[candidato]
PRESUPUESTO = 0.25                                                    # EQUIPO: su presupuesto o sus costos
u_pres = umbral_presupuesto(s_va_cand, PRESUPUESTO)
u_cost = umbral_costo(y_va, s_va_cand, costo_fp=1, costo_fn=5)
for nombre, u in [("presupuesto 25 %", u_pres), ("costos FP:FN = 1:5", u_cost)]:
    a = s_va_cand >= u
    print(f"{candidato} · {nombre:20s} umbral {u:.4f} · alertas {a.mean():.1%} · precision {precision_score(y_va, a):.3f} · recall {recall_score(y_va, a):.3f}")

### 4b · Test, una sola vez
Cuando ya eligieron modelo **y** umbral con validación, cambien `REPORTE_FINAL` a `True` y ejecuten esta celda **una vez**. Si después cambian algo por lo que vieron aquí, ya no es una evaluación honesta: díganlo en el reporte.

In [ ]:
REPORTE_FINAL = False          # EQUIPO: cámbienlo a True solo al final
UMBRAL_ELEGIDO = u_pres        # EQUIPO: el que justificaron arriba

if REPORTE_FINAL:
    modelo, _, escalado = elegidos[candidato]
    A_te = Z_te if escalado else X_te
    s_te = modelo.predict_proba(A_te)[:, 1] if hasattr(modelo, "predict_proba") else modelo.decision_function(A_te)
    a = s_te >= UMBRAL_ELEGIDO
    print(f"TEST · {candidato}: AP = {average_precision_score(y_te, s_te):.3f} · alertas = {a.mean():.1%} · "
          f"precision = {precision_score(y_te, a):.3f} · recall = {recall_score(y_te, a):.3f}")
    print(confusion_matrix(y_te, a))
else:
    print("Test sin tocar. Cambien REPORTE_FINAL a True cuando hayan decidido todo con validación.")

## 5 · Lectura con la teoría (C5)
Expliquen los resultados con los dos marcos del módulo:
- **Sesgo y varianza:** comparen AP de train y de validación de cada familia (la tabla de rejillas las trae). ¿Quién sobreajusta? ¿Quién se queda corto? ¿Qué hiperparámetro lo movió?
- **Sesgo inductivo (VI.1):** ¿qué supone cada modelo del mundo y por qué esa apuesta le ayudó o le estorbó en este problema?

Antes de declarar un ganador, revisen si la diferencia con el segundo lugar es más grande que la variación entre semanas (o el bloque de tiempo que tenga sentido en sus datos).

In [ ]:
def bootstrap_por_bloques(y, s_a, s_b, bloques, replicas=400, semilla=0):
    # Diferencia AP(a) − AP(b) remuestreando bloques completos con reemplazo
    rng = np.random.default_rng(semilla)
    idx_bloques = [np.flatnonzero(bloques == k) for k in np.unique(bloques)]
    difs = []
    for _ in range(replicas):
        idx = np.concatenate([idx_bloques[k] for k in rng.integers(0, len(idx_bloques), len(idx_bloques))])
        difs.append(average_precision_score(y[idx], s_a[idx]) - average_precision_score(y[idx], s_b[idx]))
    return np.array(difs)


semana_va = ((va["hour"] - va["hour"].min()).dt.days // 7).to_numpy()       # EQUIPO: sus bloques de tiempo
rival = comparativa.index[1]
d = bootstrap_por_bloques(y_va, puntajes_va[candidato], puntajes_va[rival], semana_va)
print(f"{candidato} − {rival}: intervalo 90 % [{np.quantile(d, 0.05):+.3f}, {np.quantile(d, 0.95):+.3f}] · gana en {np.mean(d > 0):.0%} de las réplicas")

**Lectura de resultados:** _(escribe aquí: sesgo/varianza con números de la tabla, sesgo inductivo de cada modelo, y si la diferencia entre el primero y el segundo es real)_

## 6 · Limitaciones y riesgos (C6)
- **Limitación 1:** _(p. ej. la etiqueta es un proxy: no mide vaciado real)_
- **Limitación 2:** _(p. ej. cambio de prevalencia entre meses; eventos raros que el modelo no vio)_
- **Riesgo (ético, operativo o de datos) y su mitigación:** _(p. ej. si las alertas mandan cuadrillas solo a ciertas zonas, ¿quién se queda sin servicio? ¿cómo lo vigilarían?)_

## Teoría (T1 y T2)
Dos demostraciones cortas. Primero corran la celda de cada una para ver el resultado con números; luego llenen los huecos con su álgebra.

### T1 · La mejor constante es el promedio
Si el modelo predice siempre el mismo número $c$, ¿qué $c$ deja el menor error cuadrático promedio? Aquí lo prueban a fuerza bruta con salidas reales.

In [ ]:
# Salidas reales: estación 51, 8 a.m., entre semana, solo train (EQUIPO: prueben con otra estación u hora)
base = pd.read_parquet(RUTA_DATOS)
y = base.loc[(base["particion"] == "train") & (base["station_id"] == 51)
             & (base["hour_of_day"] == 8) & (base["is_weekend"] == 0), "departures"].to_numpy(float)

candidatos = np.linspace(y.min(), y.max(), 50)                 # 50 valores de c
J = np.array([np.mean((c - y) ** 2) for c in candidatos])      # error cuadrático promedio de cada c

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(candidatos, J, "o-", color=COLORES["w"], ms=4)
ax.axvline(y.mean(), color=COLORES["error"], lw=2.5, label=f"promedio ȳ = {y.mean():.2f}")
ax.set(xlabel="c (la constante que predecimos)", ylabel="J(c)", title=f"{len(y)} mañanas de la estación 51")
ax.legend()
plt.show()
print(f"mejor c de la rejilla: {candidatos[J.argmin()]:.2f} · promedio: {y.mean():.2f}")

**Ahora la demostración.** Llenen los huecos:
1. Escriban el costo: $J(c) = \frac{1}{m}\sum_{i=1}^{m} (c - y^{(i)})^2$.
2. Deriven respecto a $c$: $\dfrac{dJ}{dc} = $ ___
3. Igualen a 0 y despejen: $c = $ ___
4. ¿Por qué es un mínimo y no un máximo? (Pista: la segunda derivada.)

_Escriban aquí su demostración._

### T2 · Precision y recall con presupuesto fijo
Los números de la sesión 2: logística con umbral 0.5, validada en julio.

In [ ]:
TP, FP, FN, TN = 808, 458, 3519, 24975      # matriz de confusión de la sesión 2 (umbral 0.5, julio)

k = TP + FP          # alertas que mandamos
P = TP + FN          # desbalances reales
precision = TP / k
recall = TP / P

print(f"k = {k:,} alertas · P = {P:,} desbalances reales")
print(f"precision = {precision:.3f} · recall = {recall:.3f}")
print("¿recall == precision · k / P?", np.isclose(recall, precision * k / P))

**Ahora la demostración.** Llenen los huecos:
1. Cada alerta es un TP o un FP, así que $k = $ ___
2. Entonces $\text{precision} = TP \,/\, $ ___
3. $\text{recall} = TP / P$, y $TP = \text{precision} \cdot $ ___
4. Si $k$ está fijo (los viajes de camión que alcanzan), ¿qué implica para el negocio? ¿Cómo pueden atrapar más desbalances?

_Escriban aquí su demostración._

---
## Lista de verificación antes de entregar
**Notebook**
- [ ] Corre completo con *Reiniciar y ejecutar todo* (si no corre: −1 punto y 48 h para corregir).
- [ ] Todo lo aprendido (umbral del evento, escaladores) se ajusta solo con train.
- [ ] Hay al menos 3 familias de modelos, cada una con su rejilla en validación.
- [ ] Test se ejecutó una vez, al final, con `REPORTE_FINAL = True`.

**Reporte (2 páginas, PDF)**
- [ ] Pregunta, variable objetivo, unidad de predicción y límites de los datos (C1).
- [ ] Partición, cómo evitaron fugas y baseline con su número (C2).
- [ ] Tabla comparativa: modelo, hiperparámetros elegidos, AP train, AP val (C3).
- [ ] Métrica y umbral justificados con presupuesto o costos, y el resultado en test (C4).
- [ ] Lectura con sesgo/varianza y sesgo inductivo; ¿la diferencia es real? (C5).
- [ ] Dos limitaciones y un riesgo con su mitigación (C6).
- [ ] Nombres del equipo y qué hizo cada quien (una línea).